## Lab — Trace a token through the network

One prompt, one vector, twelve layers.  
Every exercise passes its result to the next — run cells in order.

In [ ]:
import os
import rootutils
import numpy as np

rootutils.setup_root(os.path.abspath(''), indicator=['.git', 'pyproject.toml'], pythonpath=True)

from src.utils import load_encoder_hparams_and_params
from src.gpt2 import gpt2, layer_norm, mha, ffn

encoder, hparams, params = load_encoder_hparams_and_params('124M', '../models')
n_head = hparams['n_head']
print('hparams:', hparams)

---
### Exercise 1 — Tokenize and build x₀

The model converts the prompt into token IDs, then looks up each ID in `params['wte']` (shape `[50257 × 768]`) to get a dense vector. A positional embedding from `params['wpe']` (shape `[1024 × 768]`) is added so the model knows each token's position. The result is `x` of shape `[n_seq × 768]`.

`x[-1]` is the last token's vector. We will follow exactly this row through all 12 layers.

**Tasks:**
1. Print each token with its position index and decoded string.
2. Build `x` by summing `params['wte'][input_ids]` and `params['wpe'][range(len(input_ids))]`.
3. Print `x[-1, :8]` and `mean(|x[-1]|)` — the starting state of the vector we will trace.

In [ ]:
PROMPT = "Alan Turing theorized that computers would"
input_ids = encoder.encode(PROMPT)

print("Tokens:")
for pos, idx in enumerate(input_ids):
    print(f"  pos={pos}  id={idx:>6}  ->  '{encoder.decode([idx])}'")

# params['wte'] shape [n_vocab x n_embd] — one row per token
# params['wpe'] shape [n_ctx  x n_embd] — one row per position
x = ...  # TODO

print(f"\nx shape: {x.shape}  (n_seq x n_embd)")
print(f"x[-1, :8]     = {np.round(x[-1, :8], 4)}")
print(f"mean(|x[-1]|) = {np.abs(x[-1]).mean():.4f}")

---
### Exercise 2 — Through one block, step by step

A transformer block applies two sub-layers with residual connections:

`x = x + mha(layer_norm(x, **ln_1))` → multi-head attention  
`x = x + ffn(layer_norm(x, **ln_2))` → feed-forward network

Apply block 0 to `x_step = x.copy()` and observe how much each sub-layer changes the last-position vector.  
Compare `mean(|mha_out[-1]|)` and `mean(|ffn_out[-1]|)` — which contributes more?

**Note:** use `x_step` (a copy) so that `x` from Exercise 1 stays intact for Exercise 3.

In [ ]:
block = params['blocks'][0]
x_step = x.copy()  # copy — x from Ex 1 must stay intact for Ex 3

print("Before block 0 (last token position):")
print(f"  x[-1, :8]         = {np.round(x_step[-1, :8], 4)}")
print(f"  mean(|x[-1]|)     = {np.abs(x_step[-1]).mean():.4f}")

# MHA sub-layer — worked example
mha_out = mha(layer_norm(x_step, **block['ln_1']), **block['attn'], n_head=n_head)
x_step = x_step + mha_out

print(f"\nAfter MHA residual:")
print(f"  mean(|x[-1]|)       = {np.abs(x_step[-1]).mean():.4f}")
print(f"  mean(|mha_out[-1]|) = {np.abs(mha_out[-1]).mean():.4f}  <- MHA contribution")

ffn_out = ...  # TODO
x_step = ... # TODO

print(f"\nAfter FFN residual:")
print(f"  mean(|x[-1]|)       = {np.abs(x_step[-1]).mean():.4f}")
print(f"  mean(|ffn_out[-1]|) = {np.abs(ffn_out[-1]).mean():.4f}  <- FFN contribution")

---
### Exercise 3 — All 12 blocks: track the magnitude

Repeat the same two sub-layer steps from Exercise 2, but now for all 12 blocks on the real `x`.
After each block, print `mean(|x[-1]|)` to see how the magnitude of the last token's vector evolves with depth.

**Question:** does the magnitude grow, shrink, or stabilize as it travels deeper?

In [ ]:
# x is the original embedding from Ex 1 (Ex 2 used x_step, a copy)
print(f"{'Block':>6}  {'mean(|x[-1]|)':>14}")
print('-' * 24)
print(f"{'init':>6}  {np.abs(x[-1]).mean():>14.4f}")

for i, block in enumerate(params['blocks']):
    mha_out = ...  # TODO: same as Ex 2, replace x_step with x
    x = x + mha_out

    ffn_out = ...  # TODO: same as Ex 2, replace x_step with x
    x = x + ffn_out

    print(f"{i:>6}  {np.abs(x[-1]).mean():>14.4f}")

---
### Exercise 4 — Project to vocabulary

After 12 blocks `x` holds the final contextual representation. Two steps remain:

1. Apply the final layer norm `params['ln_f']`.
2. Project to vocabulary logits: `x_normed[-1] @ params['wte'].T`  
   (this reuses the token embedding matrix — *weight tying*).

Apply softmax to get probabilities and print the top-5 next-token predictions.  
This is the payoff: we traced one vector from a token ID all the way to a predicted next word.

In [ ]:
# x is the final hidden state after all 12 blocks

# Final layer norm
x_normed = layer_norm(x, **params['ln_f'])

# Project last position to vocabulary logits
# Weight tying: the same wte matrix used for input embeddings is reused here
logits = ...  # TODO

# Softmax probabilities
probs = np.exp(logits - np.max(logits))
probs /= probs.sum()

top5_ids = np.argsort(logits)[-5:][::-1]
print("Top-5 next-token predictions:")
for rank, idx in enumerate(top5_ids):
    token_str = encoder.decode([int(idx)])
    print(f"  #{rank+1}  id={idx:>6}  prob={probs[idx]:.4f}  token='{token_str}'")

---
### Exercise 5 — Verify against `gpt2()`

You have manually re-implemented the entire forward pass:
embeddings → 12 transformer blocks → layer norm → projection.

Call `gpt2()` and confirm your `logits` match its output to floating-point precision.

In [ ]:
# Verify that your manually computed logits match what gpt2() returns

gpt2_logits = ...  # TODO use gpt2()

print(f"Logits match gpt2(): {np.allclose(logits, gpt2_logits)}")
print(f"Max absolute difference: {np.abs(logits - gpt2_logits).max():.2e}")